# ACE-Step API Test Notebook

This notebook demonstrates how to call the ACE-Step API endpoint at `http://127.0.0.1:8001` to generate music using two different modes:

## Two Generation Modes

### 1. **Custom Mode** (自定义模式)
- You provide your own `prompt` (caption) and `lyrics`
- Full control over the music description and lyrics content
- Best for: When you have specific lyrics or want precise control

### 2. **Simple Mode** (简单模式)
- You provide a natural language `sample_query` description
- The LLM automatically generates caption, lyrics, and metadata
- Best for: Quick inspiration or when you want AI to create everything

## Setup: Select the Correct Kernel

**Before running this notebook, make sure you're using the `.venv` virtual environment:**

### Option 1: Using Jupyter Notebook/Lab UI
1. Click on the kernel name in the top-right corner of the notebook
2. Select "Select Kernel" or "Change Kernel"
3. Choose "Python 3 (ipykernel)" or look for a kernel named "ace-step-1.5" or "ACE-Step-1.5"

### Option 2: Register the kernel (if not visible)
Run these commands in your terminal:

```bash
cd /Users/xiu/Documents/LLMApp/ACE-Step-1.5

# Install ipykernel if not already installed
uv add ipykernel --dev

# Register the kernel with Jupyter
source .venv/bin/activate
python -m ipykernel install --user --name ace-step-1.5 --display-name "Python (ACE-Step-1.5)"
```

After registering, restart Jupyter and select the "Python (ACE-Step-1.5)" kernel.

### Option 3: Verify the kernel is correct
Run the cell below to check if you're using the correct Python environment.

## Workflow:
1. Submit a generation task via `/release_task` (with either Custom or Simple mode parameters)
2. Query the task status via `/query_result` until completion
3. Download the generated audio file

In [1]:
import importlib
import acestep.core.generation.handler.init_service_offload_context
importlib.reload(acestep.core.generation.handler.init_service_offload_context)

/Users/xiu/Documents/LLMApp/ACE-Step-1.5/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.10.0 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0305 13:13:33.528000 64047 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
2026-03-05 13:13:34.442 | WARNING  | acestep.training.trainer:<module>:40 - bitsandbytes not installed. Using standard AdamW.


<module 'acestep.core.generation.handler.init_service_offload_context' from '/Users/xiu/Documents/LLMApp/ACE-Step-1.5/acestep/core/generation/handler/init_service_offload_context.py'>

In [2]:
# Verify you're using the correct Python environment
import sys
import os

print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")
print(f"\nVirtual environment check:")
if '.venv' in sys.executable or 'venv' in sys.executable:
    print("✅ Using virtual environment")
else:
    print("⚠️ Not using .venv - you may need to select the correct kernel")
    
# Check if we can import project modules
try:
    import acestep
    print(f"✅ ACE-Step module found at: {acestep.__file__}")
except ImportError:
    print("⚠️ Cannot import acestep - make sure you're using the correct kernel")

Python executable: /Users/xiu/Documents/LLMApp/ACE-Step-1.5/.venv/bin/python
Python version: 3.11.6 (main, Oct  2 2023, 22:00:51) [Clang 14.0.3 (clang-1403.0.22.14.1)]

Virtual environment check:
✅ Using virtual environment
✅ ACE-Step module found at: /Users/xiu/Documents/LLMApp/ACE-Step-1.5/acestep/__init__.py


In [3]:
import requests
import json
import time
from pathlib import Path
from urllib.parse import urlparse, parse_qs

# API configuration
API_BASE_URL = "http://127.0.0.1:8001"
RELEASE_TASK_URL = f"{API_BASE_URL}/release_task"
QUERY_RESULT_URL = f"{API_BASE_URL}/query_result"
HEALTH_CHECK_URL = f"{API_BASE_URL}/health"

print(f"API Base URL: {API_BASE_URL}")

API Base URL: http://127.0.0.1:8001


In [4]:
# Check if the API server is running
try:
    response = requests.get(HEALTH_CHECK_URL, timeout=5)
    if response.status_code == 200:
        print("✅ API server is running")
        print(f"Response: {response.json()}")
    else:
        print(f"⚠️ API server returned status code: {response.status_code}")
except requests.exceptions.ConnectionError:
    print("❌ Cannot connect to API server. Make sure the server is running at http://127.0.0.1:8001")
    print("Start the server with: uv run acestep-api")
except Exception as e:
    print(f"❌ Error checking API server: {e}")

✅ API server is running
Response: {'data': {'status': 'ok', 'service': 'ACE-Step API', 'version': '1.0', 'models_initialized': True, 'llm_initialized': True, 'loaded_model': 'acestep-v15-turbo', 'loaded_lm_model': None}, 'code': 200, 'error': None, 'timestamp': 1772669614469, 'extra': None}


In [5]:
## Example 1: Custom Mode - Provide Your Own Prompt and Lyrics

# In Custom Mode, you provide both the music description (`prompt`) and the full lyrics (`lyrics`).
lyrics = """[Verse 1]
Walking down the street
Feeling the beat
Music in my heart
A brand new start

[Chorus]
This is my song
Where I belong
Singing loud and clear
For all to hear

[Verse 2]
Dreams come alive
When we strive
Together we stand
Hand in hand"""

caption = """
Chinese hip-hop, melodic trap, male rapper with smooth R&B hooks, 808 bass, hi-hats, emotional and introspective, modern C-pop influence, 90 BPM
"""
# Optional: Additional parameters
params = {
    "prompt": caption,  # Music style description (caption)
    "lyrics": lyrics,   # Full lyrics content
    "thinking": True,   # Use 5Hz LM for enhanced quality
    # "audio_duration": 60,  # Duration in seconds (10-600)
    "bpm": 120,         # Tempo (beats per minute)
    "vocal_language": "en",  # Language code
    "audio_format": "mp3",  # Output format
    "inference_steps": 8,   # Number of diffusion steps
    "batch_size": 1         # Number of audio files to generate
}

print("Generation parameters:")
print(f"Caption: {caption}")
print(f"Lyrics: {lyrics[:100]}...")
print(f"\nOther params: {json.dumps({k: v for k, v in params.items() if k not in ['prompt', 'lyrics']}, indent=2)}")

Generation parameters:
Caption: 
Chinese hip-hop, melodic trap, male rapper with smooth R&B hooks, 808 bass, hi-hats, emotional and introspective, modern C-pop influence, 90 BPM

Lyrics: [Verse 1]
Walking down the street
Feeling the beat
Music in my heart
A brand new start

[Chorus]
Thi...

Other params: {
  "thinking": true,
  "bpm": 120,
  "vocal_language": "en",
  "audio_format": "mp3",
  "inference_steps": 8,
  "batch_size": 1
}


In [6]:
# # Optional override: customize audio duration (seconds)
# # You can change this value to any integer between 10 and 600
# custom_audio_duration = 120  # <- edit this value as needed

# if custom_audio_duration is not None:
#     if not isinstance(custom_audio_duration, int):
#         raise ValueError("custom_audio_duration must be an integer number of seconds")
#     if not 10 <= custom_audio_duration <= 600:
#         raise ValueError("custom_audio_duration must be between 10 and 600 seconds")
#     params["audio_duration"] = custom_audio_duration
#     print(f"Overriding audio_duration to: {params['audio_duration']} seconds")
# else:
#     print(f"Using default audio_duration from params: {params['audio_duration']} seconds")

In [7]:
# Submit Custom Mode generation task
print("Submitting Custom Mode generation task...")
try:
    response = requests.post(
        RELEASE_TASK_URL,
        json=params,
        headers={"Content-Type": "application/json"},
        timeout=30
    )
    
    response.raise_for_status()
    result = response.json()
    
    if result.get("code") == 200:
        task_id = result["data"]["task_id"]
        status = result["data"]["status"]
        queue_position = result["data"].get("queue_position", "N/A")
        
        print(f"✅ Task submitted successfully!")
        print(f"Task ID: {task_id}")
        print(f"Status: {status}")
        print(f"Queue Position: {queue_position}")
    else:
        print(f"❌ Error: {result.get('error', 'Unknown error')}")
        task_id = None
except requests.exceptions.RequestException as e:
    print(f"❌ Request failed: {e}")
    task_id = None
except Exception as e:
    print(f"❌ Unexpected error: {e}")
    task_id = None

Submitting Custom Mode generation task...
✅ Task submitted successfully!
Task ID: b16f5c95-9122-4a31-9de2-d20cbffb0cb2
Status: queued
Queue Position: 1


In [8]:
# Poll for Custom Mode task completion
if task_id:
    print(f"\nPolling for Custom Mode task completion (Task ID: {task_id})...")
    print("Status codes: 0 = processing, 1 = success, 2 = failed")
    
    max_attempts = 60  # Maximum polling attempts
    poll_interval = 5  # Seconds between polls
    
    for attempt in range(max_attempts):
        try:
            response = requests.post(
                QUERY_RESULT_URL,
                json={"task_id_list": [task_id]},
                headers={"Content-Type": "application/json"},
                timeout=10
            )
            
            response.raise_for_status()
            result = response.json()
            
            if result.get("code") == 200 and result.get("data"):
                task_data = result["data"][0]
                status = task_data["status"]
                
                print(f"Attempt {attempt + 1}: Status = {status}", end="")
                
                if status == 1:  # Success
                    print(" ✅ Task completed!")
                    task_result = json.loads(task_data["result"])
                    print(f"\nTask Result:")
                    print(json.dumps(task_result, indent=2))
                    break
                elif status == 2:  # Failed
                    print(" ❌ Task failed!")
                    print(f"Result: {task_data.get('result', 'No error details')}")
                    break
                else:  # Processing (0)
                    print(" ⏳ Still processing...")
                    time.sleep(poll_interval)
            else:
                print(f"⚠️ Unexpected response: {result}")
                time.sleep(poll_interval)
                
        except requests.exceptions.RequestException as e:
            print(f"⚠️ Request error: {e}")
            time.sleep(poll_interval)
        except Exception as e:
            print(f"⚠️ Error: {e}")
            time.sleep(poll_interval)
    else:
        print(f"\n⏱️ Timeout: Task did not complete within {max_attempts * poll_interval} seconds")
else:
    print("Skipping polling - no task ID available")


Polling for Custom Mode task completion (Task ID: b16f5c95-9122-4a31-9de2-d20cbffb0cb2)...
Status codes: 0 = processing, 1 = success, 2 = failed
Attempt 1: Status = 0 ⏳ Still processing...
Attempt 2: Status = 0 ⏳ Still processing...
Attempt 3: Status = 0 ⏳ Still processing...
Attempt 4: Status = 0 ⏳ Still processing...
Attempt 5: Status = 0 ⏳ Still processing...
Attempt 6: Status = 0 ⏳ Still processing...
Attempt 7: Status = 0 ⏳ Still processing...
Attempt 8: Status = 0 ⏳ Still processing...
Attempt 9: Status = 0 ⏳ Still processing...
Attempt 10: Status = 0 ⏳ Still processing...
Attempt 11: Status = 0 ⏳ Still processing...
Attempt 12: Status = 0 ⏳ Still processing...
Attempt 13: Status = 0 ⏳ Still processing...
Attempt 14: Status = 0 ⏳ Still processing...
Attempt 15: Status = 0 ⏳ Still processing...
Attempt 16: Status = 0 ⏳ Still processing...
Attempt 17: Status = 0 ⏳ Still processing...
Attempt 18: Status = 0 ⏳ Still processing...
Attempt 19: Status = 0 ⏳ Still processing...
Attempt 

In [9]:
# Download the generated audio file
if task_id and 'task_result' in locals() and task_result:
    try:
        # Extract audio file URL from result
        audio_file_url = task_result[0].get("file", "")
        
        if audio_file_url:
            # Parse the URL to get the path parameter
            parsed_url = urlparse(audio_file_url)
            audio_path = parse_qs(parsed_url.query).get("path", [None])[0]
            
            if audio_path:
                # Construct download URL
                download_url = f"{API_BASE_URL}/v1/audio?path={audio_path}"
                
                print(f"Downloading audio from: {download_url}")
                
                # Download the file
                audio_response = requests.get(download_url, timeout=60)
                audio_response.raise_for_status()
                
                # Save to file
                output_dir = Path("output")
                output_dir.mkdir(exist_ok=True)
                
                # Determine file extension from format or URL
                file_ext = params.get("audio_format", "mp3")
                output_filename = output_dir / f"custom_mode_{task_id[:8]}.{file_ext}"
                
                with open(output_filename, "wb") as f:
                    f.write(audio_response.content)
                
                print(f"✅ Audio saved to: {output_filename}")
                print(f"File size: {len(audio_response.content) / 1024:.2f} KB")
            else:
                print("⚠️ Could not extract audio path from result")
        else:
            print("⚠️ No audio file URL in result")
            
    except Exception as e:
        print(f"❌ Error downloading audio: {e}")
else:
    print("Skipping download - no completed task result available")

Skipping download - no completed task result available


## Example 2: Simple Mode - Let LLM Generate Everything

In Simple Mode, you provide a natural language description (`sample_query`), and the LLM automatically generates:
- Detailed caption/description
- Complete lyrics
- Metadata (BPM, duration, key, time signature, language)

This is perfect for quick inspiration or when you want AI to handle all the creative work.

In [10]:
# Simple Mode: Define your natural language query
# The LLM will generate caption, lyrics, and metadata automatically
simple_query = "a soft acoustic guitar ballad for a quiet evening"

simple_params = {
    "sample_query": simple_query,  # Natural language description
    "thinking": True,               # Use 5Hz LM for enhanced quality
    "audio_duration": 90,           # Duration in seconds (optional)
    "vocal_language": "en",         # Language code (optional)
    "audio_format": "mp3",          # Output format
    "inference_steps": 8,          # Number of diffusion steps
    "batch_size": 1                # Number of audio files to generate
}

print("Simple Mode - Generation parameters:")
print(f"Query: {simple_query}")
print(f"\nOther params: {json.dumps({k: v for k, v in simple_params.items() if k != 'sample_query'}, indent=2)}")

Simple Mode - Generation parameters:
Query: a soft acoustic guitar ballad for a quiet evening

Other params: {
  "thinking": true,
  "audio_duration": 90,
  "vocal_language": "en",
  "audio_format": "mp3",
  "inference_steps": 8,
  "batch_size": 1
}


In [11]:
# Submit Simple Mode generation task
print("Submitting Simple Mode generation task...")
try:
    response = requests.post(
        RELEASE_TASK_URL,
        json=simple_params,
        headers={"Content-Type": "application/json"},
        timeout=30
    )
    
    response.raise_for_status()
    result = response.json()
    
    if result.get("code") == 200:
        simple_task_id = result["data"]["task_id"]
        status = result["data"]["status"]
        queue_position = result["data"].get("queue_position", "N/A")
        
        print(f"✅ Task submitted successfully!")
        print(f"Task ID: {simple_task_id}")
        print(f"Status: {status}")
        print(f"Queue Position: {queue_position}")
    else:
        print(f"❌ Error: {result.get('error', 'Unknown error')}")
        simple_task_id = None
except requests.exceptions.RequestException as e:
    print(f"❌ Request failed: {e}")
    simple_task_id = None
except Exception as e:
    print(f"❌ Unexpected error: {e}")
    simple_task_id = None

Submitting Simple Mode generation task...
✅ Task submitted successfully!
Task ID: d25fb4b2-bd58-406b-b2da-167b08dd82d3
Status: queued
Queue Position: 1


In [12]:
# Poll for Simple Mode task completion
if simple_task_id:
    print(f"\nPolling for Simple Mode task completion (Task ID: {simple_task_id})...")
    print("Status codes: 0 = processing, 1 = success, 2 = failed")
    
    max_attempts = 60  # Maximum polling attempts
    poll_interval = 5  # Seconds between polls
    
    for attempt in range(max_attempts):
        try:
            response = requests.post(
                QUERY_RESULT_URL,
                json={"task_id_list": [simple_task_id]},
                headers={"Content-Type": "application/json"},
                timeout=10
            )
            
            response.raise_for_status()
            result = response.json()
            
            if result.get("code") == 200 and result.get("data"):
                task_data = result["data"][0]
                status = task_data["status"]
                
                print(f"Attempt {attempt + 1}: Status = {status}", end="")
                
                if status == 1:  # Success
                    print(" ✅ Task completed!")
                    simple_task_result = json.loads(task_data["result"])
                    print(f"\nTask Result:")
                    print(json.dumps(simple_task_result, indent=2))
                    break
                elif status == 2:  # Failed
                    print(" ❌ Task failed!")
                    print(f"Result: {task_data.get('result', 'No error details')}")
                    break
                else:  # Processing (0)
                    print(" ⏳ Still processing...")
                    time.sleep(poll_interval)
            else:
                print(f"⚠️ Unexpected response: {result}")
                time.sleep(poll_interval)
                
        except requests.exceptions.RequestException as e:
            print(f"⚠️ Request error: {e}")
            time.sleep(poll_interval)
        except Exception as e:
            print(f"⚠️ Error: {e}")
            time.sleep(poll_interval)
    else:
        print(f"\n⏱️ Timeout: Task did not complete within {max_attempts * poll_interval} seconds")
else:
    print("Skipping polling - no task ID available")


Polling for Simple Mode task completion (Task ID: d25fb4b2-bd58-406b-b2da-167b08dd82d3)...
Status codes: 0 = processing, 1 = success, 2 = failed
Attempt 1: Status = 0 ⏳ Still processing...
Attempt 2: Status = 0 ⏳ Still processing...
Attempt 3: Status = 0 ⏳ Still processing...
Attempt 4: Status = 0 ⏳ Still processing...
Attempt 5: Status = 0 ⏳ Still processing...
Attempt 6: Status = 0 ⏳ Still processing...
Attempt 7: Status = 0 ⏳ Still processing...
Attempt 8: Status = 0 ⏳ Still processing...
Attempt 9: Status = 0 ⏳ Still processing...
Attempt 10: Status = 0 ⏳ Still processing...
Attempt 11: Status = 0 ⏳ Still processing...
Attempt 12: Status = 0 ⏳ Still processing...
Attempt 13: Status = 0 ⏳ Still processing...
Attempt 14: Status = 0 ⏳ Still processing...
Attempt 15: Status = 0 ⏳ Still processing...
Attempt 16: Status = 0 ⏳ Still processing...
Attempt 17: Status = 0 ⏳ Still processing...
Attempt 18: Status = 0 ⏳ Still processing...
Attempt 19: Status = 0 ⏳ Still processing...
Attempt 

In [13]:
# Download the Simple Mode generated audio file
if simple_task_id and 'simple_task_result' in locals() and simple_task_result:
    try:
        # Extract audio file URL from result
        audio_file_url = simple_task_result[0].get("file", "")
        
        if audio_file_url:
            # Parse the URL to get the path parameter
            parsed_url = urlparse(audio_file_url)
            audio_path = parse_qs(parsed_url.query).get("path", [None])[0]
            
            if audio_path:
                # Construct download URL
                download_url = f"{API_BASE_URL}/v1/audio?path={audio_path}"
                
                print(f"Downloading audio from: {download_url}")
                
                # Download the file
                audio_response = requests.get(download_url, timeout=60)
                audio_response.raise_for_status()
                
                # Save to file
                output_dir = Path("output")
                output_dir.mkdir(exist_ok=True)
                
                # Determine file extension from format or URL
                file_ext = simple_params.get("audio_format", "mp3")
                output_filename = output_dir / f"simple_mode_{simple_task_id[:8]}.{file_ext}"
                
                with open(output_filename, "wb") as f:
                    f.write(audio_response.content)
                
                print(f"✅ Audio saved to: {output_filename}")
                print(f"File size: {len(audio_response.content) / 1024:.2f} KB")
            else:
                print("⚠️ Could not extract audio path from result")
        else:
            print("⚠️ No audio file URL in result")
            
    except Exception as e:
        print(f"❌ Error downloading audio: {e}")
else:
    print("Skipping download - no completed task result available")

Skipping download - no completed task result available


## Example 3: Format Enhancement (Optional)

Use `use_format=true` to let the LM enhance your provided caption and lyrics. This is useful when you want to improve or normalize your input before generation.

In [14]:
# Format Enhancement Mode: Let LM enhance your provided caption and lyrics
format_params = {
    "prompt": "jazz piano",
    "lyrics": "Walking down the street, feeling the beat",
    "use_format": True,  # Let LM enhance the caption and lyrics
    "thinking": True,
    "audio_format": "mp3",
    "inference_steps": 8,
    "batch_size": 1
}

print("Format Enhancement Mode - Parameters:")
print(json.dumps(format_params, indent=2))
print("\nNote: Uncomment the code below to run this example")
# Uncomment below to run:
# response = requests.post(RELEASE_TASK_URL, json=format_params, headers={"Content-Type": "application/json"})
# print(response.json())

Format Enhancement Mode - Parameters:
{
  "prompt": "jazz piano",
  "lyrics": "Walking down the street, feeling the beat",
  "use_format": true,
  "thinking": true,
  "audio_format": "mp3",
  "inference_steps": 8,
  "batch_size": 1
}

Note: Uncomment the code below to run this example
